In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../')))

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.base import clone
from lib.preprocessor import Preprocessor

import pandas as pd
import joblib

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 100)

models = {
    'logistic regression':OneVsRestClassifier(LogisticRegression()),
    'Decision tree': DecisionTreeClassifier(),
    'multi nomial': MultinomialNB()
}

grid_params_dict = {
    'logistic regression':{
        'estimator__solver': ['liblinear'],
        'estimator__penalty': ['l1', 'l2'],
    },
    'Decision tree':{
            'criterion': ['gini', 'entropy'],
            'max_depth': [None, 2, 4, 6, 8, 10],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        },
    'multi nomial' :{
    'alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 10.0],
    'fit_prior': [True, False]
    }
}

datasets_paths = joblib.load("../data/datasets_paths.pkl")


X       = joblib.load(datasets_paths[3][0])
y_train = joblib.load('../data/train_labels.pkl')


best_estimators = {}
for key, val in models.items():
    best_estimators[key] = None

for esti_name, param_grid in grid_params_dict.items():

    grid_search = GridSearchCV(
        estimator=models[esti_name],
        param_grid=param_grid,
        scoring='roc_auc_ovr',
        cv=5,
        n_jobs=-1
    )
    grid_search.fit(X, y_train)

    print(f"{esti_name}:")
    print(f"Best Alpha: {grid_search.best_params_}")
    print(f"Best Score: {grid_search.best_score_:.4f}")
    print("_________________________________________________")
    best_estimators[esti_name] = {"model":grid_search.best_estimator_, "params":grid_search.best_params_}

del X

df_test = pd.read_csv('../data/test.csv')
X_test = df_test["tweet"].values.tolist()
y_test = df_test["sentiment"].values.tolist()


results_dict = {
    'model':[],
    'preprocessing_method':[],
    'accuracy':[],
    'AUC':[],
    'best_hyper_params':[]
}


for path, approache in datasets_paths:
        
    X_train    = joblib.load(path)
    vectorizer = approache[0]
    
    X_test_processed, _ = Preprocessor.process(
        raws=X_test,
        processing_params={
            "method": None if approache[1] == 'only' else approache[1],
            },
            show_trans=False,
            vectorizer=vectorizer
    )
    
    for name, pair in best_estimators.items():
    
        copy_model = clone(pair["model"])
        copy_model.fit(X_train, y_train)
    
        y_pred = copy_model.predict(X_test_processed)
        y_prob = copy_model.predict_proba(X_test_processed)
        acc = accuracy_score(y_pred=y_pred, y_true=y_test)
        auc = roc_auc_score(y_test, y_prob, multi_class="ovr")

        results_dict['model'].append(name)
        results_dict['preprocessing_method'].append(f"{vectorizer.__class__.__name__} + {approache[1]}")
        results_dict['accuracy'].append(acc)
        results_dict['AUC'].append(auc)
        results_dict['best_hyper_params'].append(str(pair["params"]))


results_table = pd.DataFrame(data=results_dict)
print(results_table)


logistic regression:
Best Alpha: {'estimator__penalty': 'l2', 'estimator__solver': 'liblinear'}
Best Score: 0.9772
_________________________________________________
